# Part A - FinBERT headline sentiment (run in Google Colab)
1. **Runtime -> Change runtime type -> T4 GPU**, then run all cells.
2. When asked, upload `data/news_filtered.csv` from the project.
3. Download `finbert_scores.csv` at the end and put it in the project's `data/` folder, then knit the report.
   The report adds FinBERT features (`fb_sent_mean`, `fb_neg_share`, `fb_pos_share`) to `features_daily.csv`.

In [ ]:
!pip -q install transformers datasets

In [ ]:
import pandas as pd, numpy as np, torch
from transformers import pipeline
from google.colab import files
up = files.upload()                       # choose data/news_filtered.csv
news = pd.read_csv(next(iter(up)), dtype=str, keep_default_na=False)
print(len(news), "headlines"); news.head()

In [ ]:
dev = 0 if torch.cuda.is_available() else -1
print("GPU" if dev == 0 else "CPU (slow) - switch the runtime to a GPU")
finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, device=dev, truncation=True, max_length=64)
def score(texts, bs=256):
    out = finbert(list(texts), batch_size=bs)
    return pd.DataFrame([{d["label"]: d["score"] for d in r} for r in out])[["positive", "negative", "neutral"]]

## Accuracy check on labeled financial text (Twitter Financial News Sentiment, validation split)

In [ ]:
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
tw = load_dataset("zeroshot/twitter-financial-news-sentiment")["validation"].to_pandas()
tw["truth"] = tw["label"].map({0: "negative", 1: "positive", 2: "neutral"})   # 0 Bearish, 1 Bullish, 2 Neutral
pred = score(tw["text"]).idxmax(axis=1)
acc, f1 = accuracy_score(tw["truth"], pred), f1_score(tw["truth"], pred, average="macro")
print(f"accuracy = {acc:.3f}, macro-F1 = {f1:.3f}  (n = {len(tw)})")
print(classification_report(tw["truth"], pred))
pd.DataFrame({"dataset": ["twitter-financial-news-sentiment (validation)"], "n": [len(tw)],
              "accuracy": [acc], "macro_f1": [f1]}).to_csv("finbert_accuracy.csv", index=False)

## Score every headline

In [ ]:
titles = news["Article_title"].astype(str)
uniq = pd.Series(titles.unique())                     # score each distinct headline once
s = score(uniq.tolist()); s.insert(0, "Article_title", uniq.values)
out = news[["Stock_symbol", "Date", "Article_title"]].merge(s, on="Article_title", how="left")
out = out.rename(columns={"positive": "fb_pos", "negative": "fb_neg", "neutral": "fb_neu"})
out["fb_label"] = out[["fb_pos", "fb_neg", "fb_neu"]].idxmax(axis=1).str.replace("fb_", "", regex=False)
out.to_csv("finbert_scores.csv", index=False)
print(out["fb_label"].value_counts(normalize=True).round(3))
files.download("finbert_scores.csv"); files.download("finbert_accuracy.csv")